In [1]:
import utils
import dataset
from CandleStream import CandleStream 
import requests
import json
from datetime import datetime, timedelta

In [2]:
stream = CandleStream()

[I 251113 13:32:47 smartConnect:121] in pool


In [3]:
##global parameters
symbol = 'NIFTY'
expiry = None
premium_gap = 50


In [4]:
def download_scrip_master_v1():
    url = "https://margincalculator.angelbroking.com/OpenAPI_File/files/OpenAPIScripMaster.json"
    response = requests.get(url)

    if response.status_code == 200:
        scrips = response.json()
        return scrips
    else:
        raise Exception(f"Failed to fetch data: {response.status_code}")
    
def get_token_list(scrip, symbol:str) ->tuple[dict, list]:
    """ It will return all the dictionary of tokens and with list of expiry"""
    token_dict = {}
    listofexpiry = set()
    for elem in scrip:
        if elem['instrumenttype'] == 'OPTIDX' and elem['exch_seg'] == 'NFO' and elem['name'] == symbol:
            token_dict[elem['symbol']] = elem['token']
            listofexpiry.add(elem['expiry'])

    listofexpiry = list(listofexpiry)
    listofexpiry = sorted(listofexpiry)
    return token_dict, listofexpiry


scrip = download_scrip_master_v1()

In [5]:
def weekly_expiry(listofexpiry : list) -> datetime.date:
    """It will return the closest expiry to trade"""
    todaydate = datetime.today().date()
    converted_dates = [datetime.strptime(date, "%d%b%Y").date() for date in listofexpiry]
    maxgap = 100000
    weekly_expiry_date = None
    for date in converted_dates:
        if date >= todaydate:
            if maxgap > (date - todaydate).days:
                maxgap = (date - todaydate).days
                weekly_expiry_date = date
    weekly_expiry_date = weekly_expiry_date.strftime("%d%b%Y").upper()
    year = weekly_expiry_date[-2:]
    return weekly_expiry_date[:-4] + year

In [6]:
def at_the_money_premium(symbol : str, index_price : int, expiry : str) -> tuple[str, str]:
    """This will return a tuple of ce and pe premium symbols at the money."""
    index_price = int(index_price)
    rem = index_price % premium_gap
    premium_price = index_price - rem
    premium_symbol_ce = symbol + expiry + str(premium_price) + 'CE'
    premium_symbol_pe = symbol + expiry + str(premium_price + premium_gap) + 'PE'
    if rem == 0:
        premium_symbol_pe = symbol + expiry + str(premium_price) + 'PE'

    return premium_symbol_ce, premium_symbol_pe

In [71]:
def get_index_data_in_realtime(exchange, symbol, date, interval):
    token = utils.get_token_for_index(exchange, symbol)
    df = dataset.get_data(stream, 'NSE', symbol, token, date - timedelta(1), date, interval, offset='5min') 
    df['day'] = df['timestamp'].dt.date
    df = utils.filter_data_by_dates(df.copy(), date, date)
    return df

def get_premium_data_in_realtime(exchange, symbol,token, date, interval):
    df = dataset.get_data(stream, exchange, symbol, token, date - timedelta(1), date, interval, offset='5min') 
    df['day'] = df['timestamp'].dt.date
    df = utils.filter_data_by_dates(df.copy(), date, date)
    return df

def fit_trade_rule(df):
    df['isgoodclose'] = df['close'] > df['high'].shift(1)
    df['cond1'] = (df['isgoodclose'].shift(2) == False) & (df['isgoodclose'].shift(3) == False) & (df['isgoodclose'].shift(1))
    df['breakout'] = (df['high'] > df['high'].shift(1)) & (df['cond1'].shift(1))
    if df.iloc[-1]['breakout']:
        return df.iloc[-1]
    return None

In [68]:
token_dict, listofexpiry = get_token_list(scrip, symbol)
expiry = weekly_expiry(listofexpiry)

In [69]:
today_date = datetime.today()
interval = "10min"
exchange = 'NSE'

In [73]:
indexdf = get_index_data_in_realtime(exchange, symbol, today_date, interval)
currprice = int(indexdf.iloc[-1]['close'])
premium_symbol_ce, premium_symbol_pe = at_the_money_premium(symbol, currprice, expiry)
premium_symbol_ce_token, premium_symbol_pe_token =  token_dict[premium_symbol_ce], token_dict[premium_symbol_pe]
premium_symbol_ce_token, premium_symbol_pe_token
print(premium_symbol_ce)
buy_df = get_premium_data_in_realtime('NFO', premium_symbol_ce, premium_symbol_ce_token, today_date, interval)
print(fit_trade_rule(buy_df))

sell_df = get_premium_data_in_realtime('NFO', premium_symbol_pe, premium_symbol_pe_token, today_date, interval)
print(fit_trade_rule(sell_df))



NSE NIFTY 99926000 2025-11-30 23:59:59
2025-11-30 23:59:59
Syncing for NIFTY from 2025-11-01 00:00:00 to 2025-11-30 23:59:59
NIFTY18NOV2525950CE
NFO NIFTY18NOV2525950CE 44297 2025-11-30 23:59:59
2025-11-30 23:59:59
Syncing for NIFTY18NOV2525950CE from 2025-11-01 00:00:00 to 2025-11-30 23:59:59
None
NFO NIFTY18NOV2526000PE 44304 2025-11-30 23:59:59
2025-11-30 23:59:59
Syncing for NIFTY18NOV2526000PE from 2025-11-01 00:00:00 to 2025-11-30 23:59:59
None


In [56]:
premium_symbol_ce, premium_symbol_pe

('NIFTY18NOV2525950CE', 'NIFTY18NOV2526000PE')